In [ ]:
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split



train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomCrop(224),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_folder = ImageFolder(root=r'C:\github\dataset\archive\train',transform=transforms)
train_amount = int(0.8 * len(train_folder))
val_amount = len(train_folder) - train_amount

train_set,val_set = random_split(train_folder,[train_amount,val_amount])

train_loader = DataLoader(
    train_set,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(val_set,num_workers=4,batch_size=128,shuffle=True)

In [2]:
import torch
import torch.nn as nn
import numpy as np
import torchvision

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)


In [4]:
num_classes = 2

model.conv1 = nn.Conv2d(in_channels=3, out_channels=64,kernel_size=3,stride=1,padding=1,bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.fc.in_features,num_classes)
)
model = model.to(device)


In [5]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 10

In [6]:
print(len(train_loader), len(val_loader))

2500 625


In [7]:
print(device)

cuda


In [8]:
print(device)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

cuda
True
NVIDIA GeForce RTX 4080 Laptop GPU


In [20]:
early_stopping_hyperparam = 10
early_stopping_epoch = 0
best_val_loss = None


for e in range(999):
    print(f'Epochs {e}:')
    model.train()
    print('TRAINING...')

    total_train_loss = 0
    total_train_acc =0
    train_correct = 0
    num_batches = 0
    for batch in train_loader:
        images, labels = batch
        images, labels = images.to(device), labels.to(device)
        predictions = model(images)
        loss = loss_fn(predictions, labels)
        predicted_classes = torch.argmax(predictions, dim=1)  # FIX: add dim=1
        correct = (predicted_classes == labels).sum().item()
        
        num_batches += 1
        total_train_acc += correct
        total_train_loss += loss.item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Training Loss: {total_train_loss/num_batches}')
    print(f'Training Acc: {(total_train_acc/(num_batches *128)) * 100:.2f}%')

    model.eval()
    print('VALIDATING...')
    total_val_loss = 0
    total_val_acc =0
    val_correct = 0
    num_batches = 0
    with torch.no_grad():
        for val_batch in val_loader:
            images, labels = val_batch
            images, labels = images.to(device), labels.to(device)
            predictions = model(images)

            predicted_classes = torch.argmax(predictions, dim=1)  # FIX: add dim=1
            correct = (predicted_classes == labels).sum().item()

            num_batches += 1
            total_val_acc += correct
            total_val_loss += loss_fn(predictions, labels).item()


    print(f'Val Loss: {total_val_loss/num_batches}')
    print(f'Val Acc: {(total_val_acc/(num_batches * 128)) * 100:.2f}%')
    if not best_val_loss or total_val_loss < best_val_loss:
        best_val_loss = total_val_loss
        early_stopping_epoch = 0
        print("Saving model")
        torch.save(model.state_dict(), 'model/best_model.pth')
    else:
        early_stopping_epoch += 1
        if early_stopping_epoch >= early_stopping_hyperparam:
            print('Early stopping triggered. Finish Traning.')
            break




TRAINING...
Training Loss: 0.08912315000249073
Training Acc: 24.16%
VALIDATING...
Val Loss: 0.11102050286978483
Val Acc: 23.99%
Saving model
TRAINING...
Training Loss: 0.07308997955475934
Training Acc: 24.31%
VALIDATING...
Val Loss: 0.10270393679440022
Val Acc: 24.03%
Saving model
TRAINING...
Training Loss: 0.05436058764156187
Training Acc: 24.49%
VALIDATING...
Val Loss: 0.10310731084123254
Val Acc: 24.12%
TRAINING...
Training Loss: 0.04188740637080628
Training Acc: 24.62%
VALIDATING...
Val Loss: 0.11657497135754674
Val Acc: 24.04%
TRAINING...
Training Loss: 0.033545919737865915
Training Acc: 24.69%
VALIDATING...


KeyboardInterrupt: 